# 5장 Text Clustering and Topic Modeling — SageMaker AI Studio(JupyterLab)용

<!-- origin-notice -->
> **원본 노트북:** [chapter05/Chapter 5 - Text Clustering and Topic Modeling.ipynb](https://github.com/HandsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter05/Chapter%205%20-%20Text%20Clustering%20and%20Topic%20Modeling.ipynb)  
> **원본 저장소:** [HandsOnLLM/Hands-On-Large-Language-Models](https://github.com/HandsOnLLM/Hands-On-Large-Language-Models) (Apache License 2.0, © 2024 Jay Alammar & Maarten Grootendorst)  
> 이 노트북은 원본을 SageMaker AI Studio에서 실행되도록 수정하고 한국어 설명·보충 실험을 추가한 것입니다. 변경 내용은 노트북 안의 "책과 다를 수 있는 점"에 적혀 있습니다.

원본: HandsOnLLM/Hands-On-Large-Language-Models `chapter05` 노트북(Apache 2.0, 이 노트북은 그 내용을 수정·보충한 것). 권장 환경은 1·2장과 같은 space(`ml.g4dn.xlarge`, 스토리지 30GB 이상)입니다. 4절의 텍스트 생성 단계에서 Phi-3를 쓰면 GPU가 필요하고, 나머지는 CPU로도 돌아가지만 느립니다.

| 절 | 하는 일 | 책 위치(PDF 쪽) |
|---|---|---|
| 1 | ArXiv NLP 논문 초록 44,949건 불러오기 | 193–194 |
| 2 | 군집화 3단계: 임베딩 → 차원 축소(UMAP) → 군집화(HDBSCAN), 결과 살펴보기 (+ min_cluster_size 비교) | 194–203 |
| 3 | BERTopic으로 군집을 **토픽**(키워드로 설명된 군집)으로 바꾸기, 시각화 (+ 이상치 재배정) | 204–220 |
| 4 | 토픽 표현 바꾸기: KeyBERTInspired, MMR, 텍스트 생성(Flan-T5 / Phi-3), OpenAI (+ Amazon Bedrock) | 220–231 |
| 5 | DataMapPlot으로 상위 20개 토픽 지도 그리기, 워드 클라우드 | 232 |
| 6 | (+ 보충) 자율주행 이벤트 보고서 군집화 | — |

`(+ …)` 표시는 책에 없는 보충 실험입니다. 셀은 위에서부터 차례로 실행하세요.

**용어.** 군집화(clustering)는 비슷한 문서끼리 묶는 일이고, 토픽 모델링(topic modeling)은 그렇게 묶인 군집마다 "무엇에 관한 묶음인지"를 키워드나 짧은 이름으로 붙이는 일입니다. 이 장은 앞의 것을 먼저 하고, 뒤의 것을 그 위에 얹습니다.

## 0. 환경 점검과 패키지 설치

이미지에 없는 패키지만 설치합니다. `bertopic`·`umap-learn`·`hdbscan`·`datamapplot`은 최신 버전이 numpy 2.x와 함께 동작합니다(제 작업 환경에서 bertopic 0.17.4, umap-learn 0.5.12, datamapplot 0.7.3, numpy 2.2로 확인).

`SAMPLE`을 숫자로 바꾸면 그 개수만큼만 써서 빠르게 돌려 볼 수 있습니다. 책과 같은 결과를 보려면 `None`(전체)으로 두세요.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "GPU 없음 (CPU로 진행)"
!df -h ~ | tail -1
import importlib, subprocess, sys
need = {"bertopic": "bertopic", "umap": "umap-learn", "hdbscan": "hdbscan", "datamapplot": "datamapplot",
        "datasets": "datasets", "sentence_transformers": "sentence-transformers", "wordcloud": "wordcloud",
        "openai": "openai"}
missing = [pkg for mod, pkg in need.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("설치:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("추가 설치 불필요")
import torch, transformers, numpy, bertopic
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(), "| transformers", transformers.__version__,
      "| numpy", numpy.__version__, "| bertopic", bertopic.__version__)

SAMPLE = None   # 예: 5000 으로 바꾸면 앞 5,000건만 사용

## 1. ArXiv 논문 초록 불러오기 (PDF 193–194)

1991~2024년 arXiv cs.CL(Computation and Language) 분야 논문 44,949건의 제목과 초록입니다. 레이블이 없는 데이터라, 사람이 분류 기준을 정하지 않고 비슷한 것끼리 묶는 비지도(unsupervised) 방법이 필요합니다.

In [ ]:
from datasets import load_dataset
dataset = load_dataset("maartengr/arxiv_nlp")["train"]
if SAMPLE:
    dataset = dataset.select(range(SAMPLE))

abstracts = list(dataset["Abstracts"])
titles = list(dataset["Titles"])
print(f"{len(abstracts):,}건")
print(titles[0]); print(abstracts[0][:300], "...")

## 2. 텍스트 군집화의 공통 파이프라인 (PDF 194–203)

책이 제시하는 3단계는 다음과 같습니다.

1. **임베딩:** 문서마다 벡터 하나(2장 4절의 텍스트 임베딩).
2. **차원 축소:** 384차원 벡터를 5차원으로 줄여 군집화가 잘 되게 함. 고차원에서는 점들 사이 거리가 비슷비슷해져 군집을 찾기 어렵습니다(차원의 저주).
3. **군집화:** 줄어든 공간에서 밀집된 영역을 군집으로 묶음.

### 2-1. 문서 임베딩 (PDF 195)

책은 MTEB 리더보드에서 군집화 점수가 괜찮으면서 작고 빠른 모델로 `thenlper/gte-small`을 골랐습니다(2장의 all-mpnet-base-v2보다 군집화 성능이 좋고 더 빠름). 384차원 벡터를 만듭니다. T4 GPU에서 1~2분, CPU에서는 수십 분 걸릴 수 있습니다.

In [ ]:
from sentence_transformers import SentenceTransformer

# CPU에서는 float32로 불러옴(아래 "책과 다를 수 있는 점" 참고). GPU에서는 책 코드와 같음.
dtype_key = "dtype" if int(transformers.__version__.split(".")[0]) >= 5 else "torch_dtype"
st_kwargs = {} if torch.cuda.is_available() else {"model_kwargs": {dtype_key: torch.float32}}

embedding_model = SentenceTransformer('thenlper/gte-small', **st_kwargs)
embeddings = embedding_model.encode(abstracts, show_progress_bar=True)

> **책과 다를 수 있는 점.** transformers 5.x와 최신 sentence-transformers는 모델 파일에 저장된 정밀도(gte-small은 float16)를 그대로 불러옵니다. GPU에서는 문제가 없지만 CPU에서는 float16 연산이 매우 느려, 위 셀은 GPU가 없을 때만 float32로 불러오게 했습니다. 제 작업 환경(CPU)에서 초록 200건 인코딩이 float16 22.9초, float32 8.4초였습니다.

In [ ]:
embeddings.shape   # 책: (44949, 384)

### 2-2. 차원 축소: UMAP (PDF 196–198)

**관찰 포인트:** `min_dist=0.0`은 점들을 촘촘히 모아 군집 경계를 뚜렷하게 하고, `metric='cosine'`은 고차원 임베딩 비교에 맞는 거리입니다. 책 198쪽 설명대로 `random_state=42`를 주면 결과를 재현할 수 있지만 병렬 처리가 꺼져 느려집니다.

In [ ]:
from umap import UMAP

# 384차원 → 5차원
umap_model = UMAP(n_components=5, min_dist=0.0, metric='cosine', random_state=42)
reduced_embeddings = umap_model.fit_transform(embeddings)

### 2-3. 군집화: HDBSCAN (PDF 199–200)

HDBSCAN은 밀도 기반 군집화입니다. k-means와 달리 군집 수를 미리 정하지 않고, 어느 군집에도 속하지 않는 문서를 **이상치(outlier, 레이블 -1)**로 남깁니다. `min_cluster_size=50`은 "50편 미만의 묶음은 군집으로 보지 않는다"는 뜻입니다. 책에서는 156(-1 포함)이 나왔습니다.

In [ ]:
from hdbscan import HDBSCAN

hdbscan_model = HDBSCAN(min_cluster_size=50, metric='euclidean', cluster_selection_method='eom').fit(reduced_embeddings)
clusters = hdbscan_model.labels_
len(set(clusters))

**(+ 보충)** `min_cluster_size`가 군집 수와 이상치 비율을 어떻게 바꾸는지 비교합니다. 책 200쪽 "군집을 더 만들려면 min_cluster_size를 줄이라"는 설명을 숫자로 확인하는 실험입니다.

In [ ]:
import numpy as np, pandas as pd
rows = []
for m in [25, 50, 100, 200]:
    lab = HDBSCAN(min_cluster_size=m, metric='euclidean', cluster_selection_method='eom').fit(reduced_embeddings).labels_
    rows.append({"min_cluster_size": m, "clusters": len(set(lab)) - (1 if -1 in lab else 0),
                 "outlier_ratio": round(float(np.mean(lab == -1)), 3)})
pd.DataFrame(rows)

### 2-4. 군집 살펴보기 (PDF 201–203)

군집 0의 앞 세 문서를 직접 읽어 봅니다. 군집 번호는 실행 환경에 따라 책과 다를 수 있지만, 같은 주제의 초록이 모여 있는지는 확인할 수 있습니다.

In [ ]:
cluster = 0
for index in np.where(clusters == cluster)[0][:3]:
    print(abstracts[index][:300] + "... \n")

시각화를 위해 원래 임베딩을 2차원으로 다시 줄입니다. 군집화는 5차원에서 했고, 2차원은 그림용입니다.

In [ ]:
reduced_embeddings = UMAP(n_components=2, min_dist=0.0, metric='cosine', random_state=42).fit_transform(embeddings)

df = pd.DataFrame(reduced_embeddings, columns=["x", "y"])
df["title"] = titles
df["cluster"] = [str(c) for c in clusters]

clusters_df = df.loc[df.cluster != "-1", :]
outliers_df = df.loc[df.cluster == "-1", :]

> **책과 다를 수 있는 점.** 책 202쪽 코드는 위 두 변수를 `to_plot`, `outliers`로 만들고, 바로 다음 그림 코드에서는 `clusters_df`, `outliers_df`를 씁니다. 책 코드를 그대로 치면 `NameError`가 납니다. 여기서는 공식 노트북의 이름(`clusters_df`, `outliers_df`)으로 통일했습니다.

회색 점이 이상치, 색 점이 군집입니다.

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 6))
plt.scatter(outliers_df.x, outliers_df.y, alpha=0.05, s=2, c="grey")
plt.scatter(clusters_df.x, clusters_df.y, c=clusters_df.cluster.astype(int), alpha=0.6, s=2, cmap='tab20b')
plt.axis('off'); plt.title("ArXiv cs.CL clusters (grey = outliers)")
plt.show()

## 3. 군집에서 토픽으로: BERTopic (PDF 204–220)

BERTopic은 2절의 파이프라인 뒤에 두 단계를 더 붙입니다.

- **CountVectorizer:** 군집마다 문서를 하나로 합쳐 단어 빈도를 셉니다(1장의 bag-of-words).
- **c-TF-IDF(class-based TF-IDF):** 그 군집에 자주 나오면서 다른 군집에는 드문 단어에 높은 점수를 줍니다. 이 점수 상위 단어가 토픽의 키워드가 됩니다.

임베딩·UMAP·HDBSCAN 자리에 다른 모델을 끼워 넣을 수 있어서, 책은 이를 레고 블록에 비유합니다. 아래 코드는 앞에서 만든 세 모델을 그대로 끼워 넣고, 임베딩도 다시 계산하지 않도록 넘깁니다.

In [ ]:
from bertopic import BERTopic

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    verbose=True
).fit(abstracts, embeddings)

토픽 목록입니다. `-1`은 이상치 묶음이고, `Name`은 상위 키워드 4개를 이어 붙인 기본 이름입니다.

In [ ]:
topic_model.get_topic_info()

토픽 0의 상위 10개 키워드와 c-TF-IDF 점수:

In [ ]:
topic_model.get_topic(0)

검색어와 가까운 토픽을 찾습니다. 반환값은 (토픽 번호 목록, 유사도 목록)입니다.

> **책과 다를 수 있는 점.** 책은 "topic modeling" 토픽이 **22번**(유사도 0.95)으로 나왔다고 쓰지만, 토픽 번호는 실행 환경과 라이브러리 버전에 따라 바뀝니다. 아래부터는 22를 직접 쓰지 않고 `find_topics`가 돌려준 첫 번째 번호(`tm_topic`)를 씁니다.

In [ ]:
found_topics, similarity = topic_model.find_topics("topic modeling")
tm_topic = found_topics[0]
print("topic:", tm_topic, "| similarity:", round(float(similarity[0]), 3))
topic_model.get_topic(tm_topic)

BERTopic 논문 자체가 이 토픽에 배정됐는지 확인합니다.

In [ ]:
title = 'BERTopic: Neural topic modeling with a class-based TF-IDF procedure'
if title in titles:
    print("BERTopic 논문의 토픽:", topic_model.topics_[titles.index(title)], "| topic modeling 토픽:", tm_topic)
else:
    print("SAMPLE로 줄인 데이터에 이 논문이 없습니다.")

**(+ 보충) 이상치 재배정.** 책 216쪽은 이상치를 줄이는 방법으로 `reduce_outliers()`를 언급만 합니다. 각 이상치 문서를 가장 가까운 토픽에 다시 배정해 보고, 이상치가 얼마나 줄어드는지 봅니다. 모델 자체는 바꾸지 않습니다. 기본 전략(`strategy="distributions"`, `threshold=0`)은 이상치를 모두 어딘가에 배정하므로 0%가 나올 수 있습니다. 애매한 문서까지 억지로 넣고 싶지 않다면 `threshold`를 올리세요.

In [ ]:
new_topics = topic_model.reduce_outliers(abstracts, topic_model.topics_)
before = np.mean(np.array(topic_model.topics_) == -1)
after = np.mean(np.array(new_topics) == -1)
print(f"이상치 비율: {before:.1%} → {after:.1%}")

### 3-1. 시각화 (PDF 219–220)

`visualize_documents`는 2절에서 만든 2차원 좌표 위에 토픽별로 색을 칠한 대화형(plotly) 그림입니다. 점에 마우스를 올리면 논문 제목이 보입니다. 이 셀부터는 결과 그림이 커서 노트북 파일 크기가 늘어납니다.

In [ ]:
fig = topic_model.visualize_documents(titles, reduced_embeddings=reduced_embeddings, width=1200, hide_annotations=True)
fig.update_layout(font=dict(size=16))

토픽별 키워드 막대그래프, 토픽 간 유사도 히트맵, 토픽 계층 구조입니다. 한 셀에서는 마지막 그림만 표시되므로 셋으로 나눴습니다.

In [ ]:
topic_model.visualize_barchart()

In [ ]:
topic_model.visualize_heatmap(n_clusters=30)

In [ ]:
topic_model.visualize_hierarchy()

## 4. 토픽 표현 바꾸기: 레고 블록 추가 (PDF 220–231)

c-TF-IDF 키워드는 빠르지만 "the", "model"처럼 뜻이 약한 단어가 섞이기 쉽습니다. BERTopic은 그 위에 **표현 모델(representation model)**을 하나 더 얹어 키워드를 고쳐 쓰게 합니다. 군집 결과는 그대로 두고 이름표만 바꾸는 단계라 다시 학습할 필요가 없습니다.

비교를 위해 원래 표현을 저장해 두고, 앞 5개 토픽의 상위 5개 키워드를 나란히 보여 주는 함수를 만듭니다.

In [ ]:
from copy import deepcopy
original_topics = deepcopy(topic_model.topic_representations_)

def topic_differences(model, original_topics, nr_topics=5):
    """Show the differences in topic representations between two models"""
    df = pd.DataFrame(columns=["Topic", "Original", "Updated"])
    for topic in range(nr_topics):
        og_words = " | ".join(list(zip(*original_topics[topic]))[0][:5])
        new_words = " | ".join(list(zip(*model.get_topic(topic)))[0][:5])
        df.loc[len(df)] = [topic, og_words, new_words]
    return df

### 4-1. KeyBERTInspired (PDF 223–225)

토픽의 대표 문서들 평균 임베딩과 후보 키워드 임베딩을 비교해, **의미상 가장 가까운** 키워드로 다시 순위를 매깁니다. 관찰 포인트: 복수형·불용어 같은 군더더기가 줄어드는지 보세요.

In [ ]:
from bertopic.representation import KeyBERTInspired
representation_model = KeyBERTInspired()
topic_model.update_topics(abstracts, representation_model=representation_model)
topic_differences(topic_model, original_topics)

### 4-2. Maximal Marginal Relevance (PDF 226–227)

MMR은 서로 너무 비슷한 키워드("summaries", "summary", "summarization")를 덜어 내고 **다양성**을 높입니다. `diversity`가 클수록 다양한 단어를 고릅니다.

> **책과 다를 수 있는 점.** 책 코드는 `diversity=0.2`, 공식 노트북은 `0.5`입니다. 여기서는 책 값을 쓰고, 공식 노트북 값으로 바꿔 비교해 볼 수 있게 했습니다.

In [ ]:
from bertopic.representation import MaximalMarginalRelevance
representation_model = MaximalMarginalRelevance(diversity=0.2)   # 공식 노트북: 0.5
topic_model.update_topics(abstracts, representation_model=representation_model)
topic_differences(topic_model, original_topics)

### 4-3. 텍스트 생성 블록 (PDF 228–230)

키워드 대신 **생성 모델이 토픽 이름을 직접 쓰게** 합니다. 프롬프트의 `[DOCUMENTS]`에는 토픽 대표 문서 4편(각 50단어로 자름)이, `[KEYWORDS]`에는 c-TF-IDF 키워드가 들어갑니다. 토픽마다 한 번만 생성하므로 문서 수가 아니라 토픽 수만큼만 모델을 부릅니다.

> **책과 다를 수 있는 점.** 책은 `pipeline("text2text-generation", model="google/flan-t5-small")`을 씁니다. transformers 5.x에서는 이 작업 유형(text2text-generation)이 삭제되어 `KeyError`가 납니다. 아래 셀은 버전을 확인해 4.x면 책 코드(Flan-T5)를, 5.x면 같은 프롬프트를 Phi-3(`text-generation`)로 실행합니다. Phi-3 쪽은 토픽 150개 안팎을 T4 GPU에서 몇 분 안에 처리합니다.

In [ ]:
from transformers import pipeline
from bertopic.representation import TextGeneration

prompt = """I have a topic that contains the following documents:
[DOCUMENTS]

The topic is described by the following keywords: '[KEYWORDS]'.

Based on the documents and keywords, what is this topic about?"""

GEN_MODEL = "microsoft/Phi-3-mini-4k-instruct"   # transformers 5.x일 때 쓸 생성 모델
if int(transformers.__version__.split(".")[0]) < 5:
    # 책 코드 그대로 (transformers 4.x)
    generator = pipeline('text2text-generation', model='google/flan-t5-small')
    representation_model = TextGeneration(generator, prompt=prompt, doc_length=50, tokenizer="whitespace")
else:
    generator = pipeline("text-generation", model=GEN_MODEL,
                         device_map="cuda" if torch.cuda.is_available() else "cpu",
                         # GPU: 저장된 정밀도 그대로(auto). CPU: float16/bfloat16이 매우 느려 float32
                         torch_dtype="auto" if torch.cuda.is_available() else torch.float32)  # 5.x에서는 dtype=
    # Phi-3 채팅 형식으로 감싸고, 짧은 이름만 쓰도록 한 줄 덧붙임
    chat_prompt = "<|user|>\n" + prompt + "\nAnswer with a short topic label only.<|end|>\n<|assistant|>\n"
    representation_model = TextGeneration(
        generator, prompt=chat_prompt, doc_length=50, tokenizer="whitespace",
        pipeline_kwargs={"max_new_tokens": 20, "do_sample": False, "return_full_text": False})

topic_model.update_topics(abstracts, representation_model=representation_model)
topic_differences(topic_model, original_topics)

생성 모델을 메모리에서 내립니다.

In [ ]:
import gc
del generator, representation_model
gc.collect(); torch.cuda.empty_cache()

### 4-4. OpenAI (PDF 230–231, 유료 API)

책은 GPT-3.5로 토픽 이름을 만듭니다. API 키가 있을 때만 실행되고, 없으면 건너뜁니다. 키는 코드에 직접 쓰지 말고 환경 변수 `OPENAI_API_KEY`로 넣으세요(노트북을 공유할 때 키가 함께 나가지 않도록). 책의 `gpt-3.5-turbo`가 계정에서 여전히 쓸 수 있는지는 확인이 필요합니다.

In [ ]:
import os
if os.environ.get("OPENAI_API_KEY"):
    import openai
    from bertopic.representation import OpenAI
    prompt = """
I have a topic that contains the following documents:
[DOCUMENTS]

The topic is described by the following keywords: [KEYWORDS]

Based on the information above, extract a short topic label in the following format:
topic: <short topic label>
"""
    client = openai.OpenAI()                       # 환경 변수에서 키를 읽음
    representation_model = OpenAI(client, model="gpt-3.5-turbo", exponential_backoff=True, chat=True, prompt=prompt)
    topic_model.update_topics(abstracts, representation_model=representation_model)
    display(topic_differences(topic_model, original_topics))
else:
    print("OPENAI_API_KEY 없음 → 건너뜀")

### 4-5. (+ 보충, 책 밖) Amazon Bedrock으로 토픽 이름 붙이기

Studio는 AWS 안에서 돌기 때문에 외부 API 키 대신 Bedrock을 쓸 수도 있습니다. BERTopic의 표현 모델을 거치지 않고, 토픽마다 키워드와 대표 문서를 Converse API로 보내 받은 이름을 `CustomName`으로 붙입니다(`set_topic_labels`). 키워드 표현은 그대로 남습니다.

**전제:** Studio 실행 역할(execution role)에 `bedrock:InvokeModel` 권한이 있고, 해당 리전에서 쓸 모델에 접근이 허용되어 있어야 합니다. `BEDROCK_MODEL_ID`에 Bedrock 콘솔 Model catalog의 모델 ID(또는 inference profile ID)를 넣으면 실행되고, 비워 두면 건너뜁니다.

In [ ]:
BEDROCK_MODEL_ID = ""      # 예: Bedrock 콘솔에서 확인한 모델 ID 또는 inference profile ID
N_LABEL = 20               # 앞에서부터 몇 개 토픽에 이름을 붙일지

if BEDROCK_MODEL_ID:
    import boto3
    brt = boto3.client("bedrock-runtime")
    labels = {}
    for t in range(N_LABEL):
        kw = ", ".join(w for w, _ in original_topics[t][:10])
        docs = "\n".join("- " + d[:300] for d in topic_model.get_representative_docs(t)[:3])
        q = (f"Documents:\n{docs}\n\nKeywords: {kw}\n\n"
             "Give a short topic label (max 6 words). Reply with the label only.")
        r = brt.converse(modelId=BEDROCK_MODEL_ID,
                         messages=[{"role": "user", "content": [{"text": q}]}],
                         inferenceConfig={"maxTokens": 30, "temperature": 0})
        labels[t] = r["output"]["message"]["content"][0]["text"].strip()
    topic_model.set_topic_labels(labels)
    display(topic_model.get_topic_info().head(N_LABEL + 1)[["Topic", "Count", "CustomName"]])
else:
    print("BEDROCK_MODEL_ID 비어 있음 → 건너뜀")

## 5. DataMapPlot으로 토픽 지도 그리기 (PDF 232)

상위 20개 토픽을 2차원 지도 위에 이름표와 함께 그립니다. 이름표는 **가장 최근에 적용한 표현**(위 4-3의 생성 결과)이 쓰입니다. `use_medoids=True`는 이름표 위치를 토픽 평균점이 아니라 실제 문서 중 가장 중심에 있는 점에 둡니다.

> **책과 다를 수 있는 점.** 책과 공식 노트북은 `label_font_size`, `label_wrap_width`, `use_medoids`를 `visualize_document_datamap`에 바로 넘깁니다. BERTopic 0.17에서는 이 인자들이 DataMapPlot으로 넘기는 사전 `datamap_kwds` 안으로 옮겨져, 그대로 쓰면 `TypeError: unexpected keyword argument 'label_font_size'`가 납니다. 아래 셀은 버전에 맞춰 두 방식 중 하나를 고릅니다.

In [ ]:
from packaging.version import Version
opts = dict(label_font_size=11, label_wrap_width=20, use_medoids=True)
if Version(bertopic.__version__) >= Version("0.17"):
    fig = topic_model.visualize_document_datamap(titles, topics=list(range(20)), reduced_embeddings=reduced_embeddings,
                                                 width=1200, datamap_kwds=opts)
else:   # 책 코드 그대로
    fig = topic_model.visualize_document_datamap(titles, topics=list(range(20)), reduced_embeddings=reduced_embeddings,
                                                 width=1200, **opts)
fig.savefig("datamapplot.png", dpi=300)

### 5-1. 보너스: 워드 클라우드 (공식 노트북)

토픽 키워드를 500개까지 늘린 뒤, "topic modeling" 토픽의 단어를 c-TF-IDF 점수 크기로 그립니다. 공식 노트북 마지막 셀과 같은 작업을 BERTopic 문서의 예시 형태로 옮겼습니다. `update_topics`가 키워드 표현을 다시 c-TF-IDF로 되돌린다는 점에 유의하세요.

In [ ]:
from wordcloud import WordCloud
topic_model.update_topics(abstracts, top_n_words=500)

def create_wordcloud(model, topic):
    text = {word: value for word, value in model.get_topic(topic)}
    wc = WordCloud(background_color="white", max_words=1000).generate_from_frequencies(text)
    plt.figure(figsize=(8, 4)); plt.imshow(wc, interpolation="bilinear"); plt.axis("off"); plt.show()

create_wordcloud(topic_model, topic=tm_topic)

## 6. (+ 보충) 자율주행 이벤트 보고서 군집화

같은 파이프라인을 주행 로그 분석에 옮겨 봅니다. 실제 차량에서는 이탈(disengagement)·급제동·센서 경고마다 짧은 텍스트 보고서가 쌓이고, 이를 사람이 일일이 분류하기 어렵습니다. 아래 48건은 **설명용으로 만든 가상 보고서**입니다(6개 유형 × 8건, 유형 정답은 군집화에 쓰지 않고 채점에만 씀).

데이터가 작아서 UMAP 이웃 수와 HDBSCAN 최소 군집 크기를 작게 바꿨습니다. 그리고 책 204쪽에서 말한 대로 군집화 블록을 k-means로 바꿔 끼운 결과도 함께 봅니다. 군집이 정답 유형과 얼마나 맞는지는 ARI(Adjusted Rand Index)로 잽니다. 관찰 포인트: 정답 유형을 모르는 상태에서 군집이 유형과 얼마나 맞는지, 그리고 c-TF-IDF 키워드만 보고 유형을 알아볼 수 있는지.

In [ ]:
reports = {
 "pedestrian": ["Pedestrian stepped onto crosswalk unexpectedly, safety driver braked hard.",
   "Child ran between parked cars, planner issued emergency stop.",
   "Jaywalker detected late at night, vehicle yielded with abrupt deceleration.",
   "Pedestrian partially occluded by bus, perception confidence dropped before braking.",
   "Person with stroller waited at curb, vehicle hesitated and stopped twice.",
   "Pedestrian crossing against signal forced manual takeover.",
   "Group of pedestrians spilled into lane after concert, slow creeping behavior.",
   "Pedestrian waved vehicle through but planner kept yielding, operator took over."],
 "lidar_sensor": ["Front lidar returned no points for 3 seconds, fallback to camera only.",
   "Lidar point cloud corrupted by heavy rain droplets on sensor cover.",
   "Roof lidar calibration drift detected after pothole impact.",
   "Lidar packet loss on ethernet link triggered sensor health warning.",
   "Lidar intensity saturated by retroreflective sign, phantom obstacle reported.",
   "Rear lidar timestamp out of sync with IMU, localization degraded.",
   "Snow accumulation on lidar dome blocked field of view.",
   "Lidar driver crashed and restarted, perception stack paused briefly."],
 "lane_keeping": ["Lane markings faded on highway, vehicle drifted toward shoulder.",
   "Construction zone with temporary yellow lines confused lane detection.",
   "Vehicle oscillated within lane on curved road at high speed.",
   "Lane center estimate jumped when merging lanes ended abruptly.",
   "Wet asphalt glare hid lane lines, lateral control disengaged.",
   "Vehicle crossed lane boundary while following map lane geometry.",
   "Double white lines misread as lane split, steering correction applied.",
   "Lane keeping lost tracking in tunnel exit due to lighting change."],
 "traffic_light": ["Traffic light state misclassified as green while it was red.",
   "Flashing yellow signal not recognized, vehicle stopped unnecessarily.",
   "Sun glare behind traffic signal caused unknown light state.",
   "Arrow signal for left turn detected late, vehicle missed the turn.",
   "Traffic light occluded by truck, planner waited for clear view.",
   "Signal map mismatch, light expected at wrong intersection position.",
   "Pedestrian countdown signal confused with vehicle traffic light.",
   "Power outage left traffic lights dark, vehicle requested remote assistance."],
 "gps_localization": ["GPS multipath in urban canyon shifted position by two meters.",
   "Localization lost in parking garage without satellite signal.",
   "HD map mismatch after road reconstruction degraded localization.",
   "GNSS receiver reported no fix after firmware update.",
   "Localization jumped when entering tunnel, dead reckoning drift grew.",
   "Map matching failed on newly opened highway ramp.",
   "IMU bias increased, localization covariance exceeded threshold.",
   "Satellite signal blocked under elevated highway, pose estimate unstable."],
 "cut_in": ["Vehicle cut in aggressively from adjacent lane, hard braking applied.",
   "Motorcycle lane splitting surprised planner, sudden lateral avoidance.",
   "Truck merged without signaling, time gap dropped below one second.",
   "Car cut in at highway on-ramp, adaptive cruise braked late.",
   "Taxi swerved into lane to pick up passenger, emergency stop.",
   "Bus pulled out from stop into our lane, yielding maneuver executed.",
   "Delivery van cut in then braked suddenly, follow distance violated.",
   "Neighboring car drifted into our lane while overtaking, horn and brake."],
}
docs_av = [d for v in reports.values() for d in v]
truth = [k for k, v in reports.items() for _ in v]
emb_av = embedding_model.encode(docs_av)

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

def fit_av(cluster_model):
    m = BERTopic(
        embedding_model=embedding_model,
        umap_model=UMAP(n_neighbors=8, n_components=5, min_dist=0.0, metric='cosine', random_state=42),
        hdbscan_model=cluster_model,                 # 군집화 블록만 바꿔 끼움
        vectorizer_model=CountVectorizer(stop_words="english"),
    ).fit(docs_av, emb_av)
    print(f"ARI = {adjusted_rand_score(truth, m.topics_):.2f}  (1: 정답과 완전히 일치, 0: 무작위 수준)")
    display(pd.crosstab(pd.Series(truth, name="truth"), pd.Series(m.topics_, name="topic")))
    return m

print("[A] HDBSCAN (min_cluster_size=4)")
av_hdb = fit_av(HDBSCAN(min_cluster_size=4, metric='euclidean', cluster_selection_method='eom'))
print(); print("[B] k-means (k=6)")
av_km = fit_av(KMeans(n_clusters=6, random_state=42, n_init=10))
av_km.get_topic_info()[["Topic", "Count", "Name"]]

**읽는 법.** 교차표의 각 행(정답 유형)이 한 열(토픽)에 몰려 있으면 군집이 유형을 잘 찾은 것입니다. 제 작업 환경(CPU, 라이브러리 버전은 0절 출력 참고)에서는 HDBSCAN이 ARI 0.56, k-means가 0.67이었습니다. 라이다·측위 유형은 두 방식 모두 정확히 묶였고, 끼어들기(cut-in)와 차선 유지(lane keeping)는 서로 섞였습니다. 두 유형 모두 `lane`이 자주 나오는 문장이라, 문장 임베딩이 원인(다른 차의 끼어들기 vs 자차의 차선 인식)보다 겉으로 비슷한 의미에 끌린 결과입니다. HDBSCAN은 데이터가 48건뿐이라 파라미터를 조금만 바꿔도 군집 수가 크게 달라집니다(min_cluster_size 3~6, n_neighbors 3~8에서 2~7개). 실제 로그처럼 수천 건 단위가 되면 책의 설정에 가까워져 안정됩니다. 정답 유형을 알려 주지 않았는데도 키워드에 `lidar`, `lane`, `signal`, `gps` 같은 단어가 떠오르는지 보세요.

## 7. 정리

| 단계 | 하는 일 | 이 노트북 |
|---|---|---|
| 임베딩 | 문서 → 벡터 1개 | 2-1 |
| 차원 축소 (UMAP) | 384차원 → 5차원(군집용), 2차원(그림용) | 2-2, 2-4 |
| 군집화 (HDBSCAN) | 밀집 영역을 군집으로, 나머지는 이상치 | 2-3 |
| c-TF-IDF | 군집마다 특징 키워드 | 3 |
| 표현 모델 | 키워드 다듬기(KeyBERTInspired, MMR) 또는 이름 생성(LLM) | 4 |

**자율주행·Agent로 이어지는 점(책 밖).** 6절처럼 이 파이프라인은 레이블 없는 주행 이벤트 보고서, 고객 불만, 장애 티켓을 묶어 "어떤 유형이 많은가"를 먼저 보는 데 그대로 쓸 수 있습니다. 텍스트 대신 이미지·장면 임베딩(9장의 CLIP 같은 멀티모달 임베딩)에 적용하면 대량의 주행 영상에서 비슷한 장면을 묶어 학습 데이터 큐레이션에 쓸 수 있습니다. 4-3·4-5처럼 군집마다 LLM이 이름을 붙이는 단계는, Agent가 대량 데이터를 요약해 사람에게 보고하는 작업의 축소판입니다.

**끝나면** Studio에서 space를 **Stop** 하세요. EBS 볼륨(모델·데이터 캐시)은 남아 다음 장에서 재사용됩니다.